## Modeling and Solving the Powertrain Design

Here is the design problem. A motor runs at 3600 rpm, and the output shaft it drives
has to turn at 300 rpm. That is a reduction of exactly 12 to 1, and we want to achieve
it with a **compound gear train**: two stages of meshing spur gears, where the second
gear of the first stage and the first gear of the second stage sit on a shared
intermediate shaft.

Four gears means four teeth counts, $N_1$ through $N_4$, and each of them has to satisfy
the same physical constraints:

- **Undercutting.** Below about 12 teeth the cutter removes material from the base of
  each tooth and the gear is weakened, so $N \geq 12$.
- **Housing size.** The gearbox has to fit, so $N \leq 80$.
- **Whole teeth.** A gear cannot have 22.5 teeth. Every count is an integer.

That last constraint is what makes this awkward by hand. A ratio of 12 is easy to hit
with real numbers and much less obvious to hit with four integers that are also bounded
above and below.

### Writing the ratio as a constraint

The overall reduction of a compound train is the product of the two stage ratios:

$$ G_{total} = \frac{N_2}{N_1} \cdot \frac{N_4}{N_3} = 12 $$

We could hand that to Z3 as written, but dividing integers drags us into rational
arithmetic for no reason. Multiplying through by $N_1 N_3$ gives an equivalent
constraint with nothing but integer multiplication in it:

$$ N_2 \, N_4 = 12 \, N_1 \, N_3 $$

That is the form we will use.

#### 1. Create a fresh solver

In [ ]:
powertrain_solver = Solver()

#### 2. Declare the variables

One integer variable per gear, holding its number of teeth.

In [ ]:
# Define integer variables for the number of teeth
N1 = Int('N1')
N2 = Int('N2')
N3 = Int('N3')
N4 = Int('N4')

#### 3. Add the design constraints

Each of the four ideas from above becomes one or two lines. Nothing here is solved or
rearranged for the solver's benefit beyond the cross-multiplication we did a moment
ago.

In [ ]:
# 1. Minimum teeth to avoid undercut (standard engineering practice)
min_teeth = 12
powertrain_solver.add(N1 >= min_teeth, N2 >= min_teeth, N3 >= min_teeth, N4 >= min_teeth)

# 2. Maximum teeth due to housing size constraints
max_teeth = 80
powertrain_solver.add(N1 <= max_teeth, N2 <= max_teeth, N3 <= max_teeth, N4 <= max_teeth)

# 3. The gear ratio constraint, cross-multiplied to stay in integers
powertrain_solver.add(N2 * N4 == 12 * N1 * N3)

# 4. Avoiding "idler" redundancy (ensure both stages contribute to reduction)
powertrain_solver.add(N2 > N1)
powertrain_solver.add(N4 > N3)

# Let's see what we have asked for
showSolver(powertrain_solver)

#### 4. Solve the system

In [ ]:
# Check if solution exists
print( powertrain_solver.check() )

In [ ]:
# View solution
solution = powertrain_solver.model()
print( solution )

"sat" means a set of teeth counts exists that meets every constraint at once, and the
model is one such set. Let's read the two stage ratios out of it and confirm they
multiply to 12.

In [ ]:
n1, n2, n3, n4 = [solution[v].as_long() for v in [N1, N2, N3, N4]]

print(f"stage 1: {n2} teeth driven by {n1}")
print(f"stage 2: {n4} teeth driven by {n3}")
print(f"total reduction: {(n2*n4)/(n1*n3)} to 1")

Here, we will visualize the **relative sizes of the gears** in the powertrain layout to
see the design we have been handed.

The two stages are drawn on separate rows because that is where they physically sit:
$N_2$ and $N_3$ share an intermediate shaft, so the second stage is offset along that
shaft rather than sitting in the same plane as the first. Within each row the spacing is
to scale, so the gap between two shaft centres really is the sum of the two pitch
radii.

In [ ]:
# Run the visualization with the solver's results
plot_gears(n1, n2, n3, n4)

#### 5. A design that will not fit

Now suppose the housing shrinks and no gear can exceed 40 teeth. Everything else about
the problem is unchanged: same motor, same output speed, same undercutting limit.

Rather than working out by hand whether 12 to 1 is still reachable, we can just state
the new bound and ask.

In [ ]:
small_housing = Solver()

N1, N2, N3, N4 = Ints('N1 N2 N3 N4')

min_teeth = 12
max_teeth = 40 # the housing has shrunk
small_housing.add(N1 >= min_teeth, N2 >= min_teeth, N3 >= min_teeth, N4 >= min_teeth)
small_housing.add(N1 <= max_teeth, N2 <= max_teeth, N3 <= max_teeth, N4 <= max_teeth)

small_housing.add(N2 * N4 == 12 * N1 * N3)
small_housing.add(N2 > N1)
small_housing.add(N4 > N3)

print( small_housing.check() )

"unsat" means no assignment of the four teeth counts satisfies all of those constraints
together. This is not a failure to find something; it is a proof that nothing is there.
With a ceiling of 40 teeth and a floor of 12, the most any single stage can reduce by is
$40/12$, a little over 3.3, so the best two stages together can manage is about 11.1.
Twelve is out of reach.

The solver has told us something useful about the design rather than about our
arithmetic: this gearbox cannot be built in that housing, and no amount of searching for
clever teeth counts will change that. Widening the housing or adding a third stage
would.